# 03 · Feature Engineering

**Goal:** turn the EDA findings into model inputs and document every choice. This is **⅓ of the grade**.

> *"Do feature engineering using patterns you discover in the data, best practices of data science and your judgement. Document and justify any new features you create and your final selection of features used in the model."*

| | |
|---|---|
| **Inputs** | `clean/inspections_clean.pkl`, `clean/violation_codes_clean.csv`, `clean/split.csv`, `raw/inspector_acronym_lookup.csv`, `raw/regulations_corpus.jsonl` |
| **Outputs** | `features/features.pkl` (ID, time, target, split + every feature), `features/feature_dictionary.csv` |

**Rules for this notebook**
1. **Row-wise only.** A feature uses that row's own data plus fixed lookups (codes, acronyms, regulations). That's safe to compute on every row.
2. **Feature construction doesn't use the target; feature selection does.** Sections 3–8 build each value from the row itself, fixed lookups or (for history) the same restaurant's earlier rows, and imputation, scaling, one-hot encoding and text vectorizing are fitted later, inside 04's CV pipelines. But the choice of which features to build and keep is **target-informed**: it follows the training-set EDA in 02, and section 9 selects features by cross-validated scores on the training labels, using the same restaurant-grouped folds as 04. So 04's CV scores for the selected features are development results and somewhat optimistic; the held-out restaurants, which nothing here touches, give the final evaluation.
3. **History features only look backwards** in time within a restaurant.
4. **Every feature is registered** with `add_feature()`: its feature set, kind, source, description and the reason you made it. The registry becomes the feature table in the report.
5. **Choices that depend on the data** (merging rare categories, screening, dropping) use training rows only.

## 0. Setup

In [1]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 90)

ROOT = Path.cwd()
RAW = ROOT / "raw"
CLEAN = ROOT / "clean"
FEAT = ROOT / "features"

TARGET = "NEXT_INSPECTION_GRADE_C_OR_BELOW"
GROUP_KEY = "RESTAURANT_SERIAL_NUMBER"
TIME_COL = "INSPECTION_TIME"
RANDOM_STATE = 42

## 1. Load

In [2]:
for path in [CLEAN / "inspections_clean.pkl", CLEAN / "split.csv"]:
    if not path.exists():
        raise FileNotFoundError(f"{path.relative_to(ROOT)} not found. Run 01_data_cleaning and 02_eda first.")

df = pd.read_pickle(CLEAN / "inspections_clean.pkl")
df[TARGET] = df[TARGET].astype(int)
codes = pd.read_csv(CLEAN / "violation_codes_clean.csv")
missing_cols = {"n_violations", "violation_list_complete", "violation_ids"} - set(df.columns)
assert not missing_cols, f"rerun 01_data_cleaning: missing {missing_cols}"
split = pd.read_csv(CLEAN / "split.csv", index_col="row")["split"]
assert split.index.equals(df.index), "split.csv doesn't line up with the cleaned data: rerun 02_eda."
is_train = split.eq("train")

acronyms = pd.read_csv(RAW / "inspector_acronym_lookup.csv", dtype=str)
with open(RAW / "regulations_corpus.jsonl", encoding="utf-8") as f:
    corpus = pd.DataFrame([json.loads(line) for line in f if line.strip()])

print(f"{len(df):,} rows ({is_train.sum():,} train / {(~is_train).sum():,} holdout) | "
      f"{len(acronyms)} acronyms | {len(corpus)} corpus docs")

35,000 rows (28,036 train / 6,964 holdout) | 48 acronyms | 239 corpus docs


## 2. Feature registry

In [3]:
features = pd.DataFrame(index=df.index)
registry = {}


def add_feature(name, values, feature_set, kind, source, description, rationale):
    """Add (or replace) one model input and document it.
    feature_set: a group name so 04 can test sets with/without (e.g. "violations", "comments", "current_status").
    kind: "numeric", "binary", "categorical", or "text" (one cleaned string per row, never NaN;
          04 turns it into TF-IDF inside the CV pipeline).
    Numeric and binary values are stored as float64 and categories as Python strings, with NaN for missing in both:
    the types scikit-learn's imputers and encoders accept."""
    values = pd.Series(values, index=df.index)
    if kind in ("numeric", "binary"):
        values = pd.to_numeric(values).astype("float64")
    elif kind == "categorical":
        values = values.astype(object).where(values.notna(), np.nan)
    features[name] = values
    registry[name] = {"feature": name, "feature_set": feature_set, "kind": kind, "source": source,
                      "description": description, "rationale": rationale, "keep": True, "drop_reason": ""}


def drop_feature(name, reason):
    """Keep a feature documented but exclude it from modeling, with the reason (it still shows in the report)."""
    registry[name].update(keep=False, drop_reason=reason)


def describe_features(names):
    """Quick look at registered numeric/binary features on training rows."""
    t = features.loc[is_train, names]
    return pd.DataFrame({"missing": t.isna().sum(), "mean": t.mean(), "min": t.min(), "max": t.max()}).round(3)


def get_fragments(frame):
    """Comment fragments per row: the list column saved by 01 if present, otherwise split the comment on '//'."""
    if "comment_fragments" in frame:
        return frame["comment_fragments"]
    return frame["Comments"].map(
        lambda t: [f.strip().lower() for f in t.split("//") if f.strip()] if isinstance(t, str) else [])

## 3. Violation features

What this inspection found is the EDA's strongest signal (findings 1–5). Every candidate below comes from the cleaned violation list and the code lookup. Several overlap (EDA section 12), so section 9 decides which of them earn a place.

**Rule for anything built from the list:** about 1% of inspections have no usable violation list (`violation_list_complete = 0`). For those rows:
- the count is `n_violations` (from `NUMBER_OF_VIOLATIONS`), never `len(violation_ids)`;
- list-based counts and totals are **missing, not 0**;
- "has an IHH code" is 1 when the named first/second/third codes confirm it, otherwise missing. An unavailable list can't prove absence.

The helpers below follow this rule, and 04's imputer adds a missing-value indicator for these gaps.

In [4]:
add_feature("n_violations", df["n_violations"], "violations", "numeric",
            source="n_violations (01: list length, or NUMBER_OF_VIOLATIONS when the list is unavailable)",
            description="Number of violations recorded at this inspection",
            rationale="EDA finding 1: the rate climbs steadily from 15.3% at 3 violations to 71.3% at 15+ with no "
                      "plateau (univariate AUC 0.626), so it stays a number rather than bins")
add_feature("violation_list_complete", df["violation_list_complete"], "violations", "binary",
            source="violation_list_complete (01)",
            description="1 = the full violation list is available; 0 = unavailable or incomplete",
            rationale="List-based features are missing (not 0) when the list is unavailable; this flag keeps that "
                      "distinction visible to the model and the reader")

In [5]:
COMPLETE = df["violation_list_complete"].eq(1)
SEV = codes.set_index("Violation_ID")["severity"].to_dict()
DEM = codes.set_index("Violation_ID")["Violation_Demerits"].to_dict()
RANK = codes.set_index("Violation_ID")["severity_rank"].to_dict()
NAMED = df[["first_violation_id", "second_violation_id", "third_violation_id"]]


def from_complete_lists(summarize):
    """summarize(list of IDs) -> number, for complete lists only. NaN where the list is unavailable or incomplete:
    a missing list is not zero violations."""
    out = pd.Series(np.nan, index=df.index)
    out[COMPLETE] = df.loc[COMPLETE, "violation_ids"].map(summarize).astype(float)
    return out


def severity_present(level):
    """1 if a code of this severity is confirmed (in the list or the named first/second/third codes),
    0 only when a COMPLETE list has none, NaN otherwise: an incomplete list can't prove absence."""
    in_list = df["violation_ids"].map(lambda ids: isinstance(ids, list) and any(SEV.get(i) == level for i in ids))
    in_named = NAMED.apply(lambda col: col.map(SEV).eq(level)).any(axis=1)
    return pd.Series(np.where(in_list | in_named, 1.0, np.where(COMPLETE, 0.0, np.nan)), index=df.index)


LIST_SOURCE = "violation_ids + code lookup (complete lists only)"
add_feature("critical_count", from_complete_lists(lambda ids: sum(SEV[i] == "Critical" for i in ids)),
            "violations", "numeric", source=LIST_SOURCE,
            description="Number of Critical codes recorded at this inspection",
            rationale="EDA finding 3: the strongest severity count; the rate rises from 14.4% with no critical code "
                      "to 57.5% with 5 or more")
add_feature("worst_severity_rank", from_complete_lists(lambda ids: max(RANK[i] for i in ids)),
            "violations", "numeric", source=LIST_SOURCE,
            description="Most severe code recorded: 0 = no-demerit codes only, 1 = Non-Major, 2 = Major, "
                        "3 = Critical, 4 = Imminent Health Hazard",
            rationale="EDA finding 2: the rate steps up in the same order as the ranks (Non-Major 10.5%, Major 14.4%, "
                      "Critical 27.5%, IHH 29.7%), so one ordered number fits")
add_feature("ihh_present", severity_present("Imminent Health Hazard"), "violations", "binary",
            source="violation_ids + first/second/third_violation_id + code lookup",
            description="1 = an Imminent Health Hazard code is confirmed; 0 = a complete list has none; "
                        "missing = no usable list and none of the named codes is an IHH",
            rationale="EDA finding 4: a confirmed hazard raises the rate to 29.8% vs 23.3%. IHH codes carry "
                      "0 demerits, so no demerit total can see them")
add_feature("code_demerits_total", from_complete_lists(lambda ids: sum(DEM[i] for i in ids)),
            "violations", "numeric", source=LIST_SOURCE,
            description="Demerits implied by every listed code (lookup demerits summed)",
            rationale="EDA finding 5: ranks risk better (AUC 0.635) than the recorded INSPECTION_DEMERITS (0.604) or "
                      "the first-three total (0.590), but overlaps n_violations (Spearman 0.83)")
add_feature("major_count", from_complete_lists(lambda ids: sum(SEV[i] == "Major" for i in ids)),
            "violations", "numeric", source=LIST_SOURCE,
            description="Number of Major codes recorded at this inspection",
            rationale="EDA finding 3: barely moves the rate until 4 or more, which looks like a proxy for the total; "
                      "a candidate only if it adds something beyond n_violations")
add_feature("non_major_count", from_complete_lists(lambda ids: sum(SEV[i] == "Non-Major" for i in ids)),
            "violations", "numeric", source=LIST_SOURCE,
            description="Number of Non-Major codes recorded at this inspection",
            rationale="EDA finding 3: same pattern as the Major count; a candidate only if it adds something")
add_feature("critical_share", from_complete_lists(lambda ids: np.mean([SEV[i] == "Critical" for i in ids])),
            "violations", "numeric", source=LIST_SOURCE,
            description="Share of the recorded codes that are Critical",
            rationale="Separates 'many small problems' from 'a few serious ones' at the same violation count")
# How often the first listed code is the most severe one (training rows, complete lists; ties count as the top)
first_has_top_rank = from_complete_lists(lambda ids: RANK[ids[0]] == max(RANK[i] for i in ids))[is_train].mean()
first_has_top_demerits = from_complete_lists(lambda ids: DEM[ids[0]] == max(DEM[i] for i in ids))[is_train].mean()
print(f"first listed code: highest severity rank in {first_has_top_rank:.1%} of complete training lists, "
      f"highest demerit value in {first_has_top_demerits:.1%}")
add_feature("first_violation_demerits", df["first_violation_demerits"], "violations", "numeric",
            source="first_violation_demerits (01: demerits of the first listed code)",
            description="Demerits of the first code on the inspection report",
            rationale=f"EDA finding 5: the strongest single demerit measure (AUC 0.641), but the first code is the "
                      f"most severe one (highest severity rank) in only {first_has_top_rank:.0%} of complete "
                      f"training lists and has the highest demerit value in {first_has_top_demerits:.0%}, so part "
                      f"of its signal may come from recording order")
add_feature("total_mapped_demerits", df["TOTAL_MAPPED_DEMERITS"], "violations", "numeric",
            source="TOTAL_MAPPED_DEMERITS (as delivered)",
            description="Demerits of the first three listed codes",
            rationale="Delivered as a ready-made demerit total, so it was considered")
drop_feature("total_mapped_demerits", "EDA finding 5: covers only the first three codes (AUC 0.590) and barely tracks "
             "the recorded demerits (Spearman 0.12); code_demerits_total covers the full list")

describe_features([n for n, r in registry.items() if r["feature_set"] == "violations"])

first listed code: highest severity rank in 63.7% of complete training lists, highest demerit value in 52.1%


,missing,mean,min,max
n_violations,6,5.350,3.0,27.0
violation_list_complete,0,0.991,0.0,1.0
critical_count,245,1.177,0.0,10.0
worst_severity_rank,245,2.741,1.0,4.0
ihh_present,238,0.053,0.0,1.0
code_demerits_total,245,14.657,2.0,70.0
major_count,245,1.933,0.0,12.0
non_major_count,245,2.184,0.0,14.0
critical_share,245,0.208,0.0,1.0
first_violation_demerits,2,3.609,0.0,10.0


## 4. Inspection context and current status

`INSPECTION_TYPE` (re-inspection vs routine) and `INSPECTION_DEMERITS` as recorded.

**Leakage decision:** `CURRENT_GRADE` / `CURRENT_DEMERITS` may describe the restaurant *after* this inspection (that's what they mean in the real SNHD feed). They're registered under their own feature set, `"current_status"`, so 04 can compare the model with and without them. The check at the end of this section adds one piece of evidence about when they were measured.

In [6]:
add_feature("inspection_demerits", df["INSPECTION_DEMERITS"], "violations", "numeric",
            source="INSPECTION_DEMERITS (as recorded)",
            description="Demerits recorded for this inspection",
            rationale="EDA finding 5: a clear gradient (16.7% at 0-5 demerits to 47.4% at 21+), yet only Spearman "
                      "0.57 with the code-implied total, so it may carry separate information")
add_feature("is_reinspection",
            df["INSPECTION_TYPE"].eq("Re-inspection").astype(float).where(df["INSPECTION_TYPE"].notna()),
            "inspection", "binary", source="INSPECTION_TYPE",
            description="1 = re-inspection, 0 = routine or other",
            rationale="A re-inspection follows a failed visit, so it could mark a struggling restaurant")
drop_feature("is_reinspection", "EDA section 7: re-inspections 24.0% (n = 1,713) vs routine 23.6%, unadjusted p = 0.74")

GRADE_RANK = {"A": 0, "B": 1, "C": 2}
add_feature("current_grade_rank", df["CURRENT_GRADE"].map(GRADE_RANK), "current_status", "numeric",
            source="CURRENT_GRADE (A/B/C; other codes set to missing in 01)",
            description="Current letter grade as an ordered number: 0 = A, 1 = B, 2 = C",
            rationale="EDA finding 7: A 21.9%, B 25.8%, C 29.9%, an ordered step. LEAKAGE RISK: its timing is "
                      "undocumented, so it has its own feature set for 04's with/without comparison")
add_feature("current_demerits", df["CURRENT_DEMERITS"], "current_status", "numeric",
            source="CURRENT_DEMERITS", description="The restaurant's current demerits",
            rationale="EDA finding 7: 18.5% at 0-2 demerits to 41.6% at 11+. LEAKAGE RISK: same timing caveat as "
                      "the grade")

# Timing evidence: some comments record the grade posted at THIS inspection ("grade a posted"). Compare it with
# CURRENT_GRADE on training rows (no target involved).
posted = get_fragments(df).map(lambda frags: next(
    (f.split()[1].upper() for f in frags if re.fullmatch(r"grade [abc] posted", f)), None))
has_posted = posted.notna() & is_train
same_grade = posted[has_posted].eq(df.loc[has_posted, "CURRENT_GRADE"])
print(f"training rows whose comment records a posted grade: {has_posted.sum():,} "
      f"({has_posted.sum() / is_train.sum():.0%}); same grade as CURRENT_GRADE: {same_grade.sum():,} "
      f"({same_grade.mean():.2%})")
pd.crosstab(posted[has_posted].rename("grade posted (comment)"), df.loc[has_posted, "CURRENT_GRADE"])

training rows whose comment records a posted grade: 8,655 (31%); same grade as CURRENT_GRADE: 8,654 (99.99%)


CURRENT_GRADE,A,B,C
grade posted (comment),,,
A,5653,0,0
B,1,2163,0
C,0,0,838


**What the posted-grade check shows:** where the inspector's comment for *this* inspection records the grade posted (about 31% of training rows), it matches `CURRENT_GRADE` in all but one row. That points to `CURRENT_GRADE` being the grade issued at this inspection, which would be known at prediction time. It isn't proof, though: both fields could have been generated from the same value when the exercise data were prepared, and EDA section 7 showed the grades don't follow the SNHD demerit cut-offs. So the two columns keep their own feature set, and the final decision stays with 04's with/without comparison. The "grade X posted" fragments themselves add nothing beyond `CURRENT_GRADE`, so section 7 doesn't use them.

## 5. Restaurant attributes

- **Seating** (EDA finding 6): a step at 200 seats, flat on either side, so a `large_venue` flag replaces raw seats.
- **Category** (finding 8): only buffets clearly differ. Both a pooled category (categories with fewer than 100 *training* rows become "Other") and a buffet flag are registered, and section 9 keeps whichever earns its place.
- **Operating hours** (finding 14): borderline over 24 schedules with no pattern. Instead of 24 one-hot columns, the schedules are parsed into four interpretable attributes for section 9 to judge.
- **Restaurant age, employees and location** (finding 14): no clear association. They're registered so the report shows they were considered, then dropped with the EDA evidence.

In [7]:
seats = df["restaurant_seating_capacity"]
add_feature("large_venue", seats.gt(200), "restaurant", "binary",
            source="restaurant_seating_capacity", description="1 = more than 200 seats",
            rationale="EDA finding 6: 30.6% above 200 seats vs 17.6% at or below, flat within each side, and the "
                      "gap holds at every violation count (+12-15 points)")
add_feature("seating_capacity", seats, "restaurant", "numeric", source="restaurant_seating_capacity",
            description="Seating capacity (25-500 in steps of 25)", rationale="The raw form of large_venue")
drop_feature("seating_capacity", "EDA finding 6: no clear difference within <=200 seats (unadjusted p = 0.58) or within >200 seats "
             "(unadjusted p = 0.16); large_venue carries the whole step in one parameter")

# Category: pool categories with fewer than MIN_CATEGORY_ROWS training rows (decided on training counts only)
MIN_CATEGORY_ROWS = 100
category = df["RESTAURANT_CATEGORY"]
train_counts = category[is_train].value_counts()
common = train_counts.index[train_counts >= MIN_CATEGORY_ROWS]
add_feature("restaurant_category", category.where(category.isin(common) | category.isna(), "Other"),
            "restaurant", "categorical", source="RESTAURANT_CATEGORY",
            description=f"Restaurant category; the {len(train_counts) - len(common)} categories with fewer than "
                        f"{MIN_CATEGORY_ROWS} training rows are pooled into 'Other'",
            rationale="EDA finding 8: buffets are lower (14.3%); the small categories are too noisy to read alone")
add_feature("is_buffet", category.eq("Buffet").astype(float).where(category.notna()), "restaurant", "binary",
            source="RESTAURANT_CATEGORY", description="1 = Buffet",
            rationale="EDA finding 8: the one category that clearly differs (14.3% [11.2, 18.2], n = 453); the other "
                      "large categories sit within 1.5 points of the average")
pooled = category.notna() & ~category.isin(common)
print(f"categories kept: {len(common)} + 'Other' ({pooled[is_train].sum()} training rows pooled); "
      f"missing: {category[is_train].isna().sum()} training rows")

add_feature("restaurant_age_years", (df[TIME_COL] - df["restaurant_open_date"]).dt.days / 365.25,
            "restaurant", "numeric", source="INSPECTION_TIME - restaurant_open_date",
            description="Years since the restaurant opened, at this inspection",
            rationale="Newer restaurants could plausibly be less practiced at compliance")
drop_feature("restaurant_age_years", "EDA finding 14: quintile rates 23.3-24.3%, AUC 0.494, unadjusted p = 0.77")

for column, auc, p in [("EMPLOYEE_COUNT", 0.497, 0.34), ("MEDIAN_EMPLOYEE_AGE", 0.500, 0.31),
                       ("MEDIAN_EMPLOYEE_TENURE", 0.502, 0.51)]:
    add_feature(column.lower(), df[column], "restaurant", "numeric", source=column,
                description=column.replace("_", " ").capitalize(),
                rationale="Staffing could plausibly affect compliance")
    drop_feature(column.lower(), f"EDA finding 14: quintile rates within 2 points, AUC {auc:.3f}, unadjusted p = {p:.2f}")

in_venue = df["RESTAURANT_LOCATION"].ne(df["RESTAURANT_NAME"]).astype(float).where(df["RESTAURANT_LOCATION"].notna())
for name, values, kind, source, description, evidence in [
    ("city", df["CITY"], "categorical", "CITY", "City", "unadjusted p = 0.66 across 6 cities"),
    ("zip5", df["ZIP5"], "categorical", "ZIP5", "5-digit ZIP code", "29 ZIPs (300+ rows) range 20.5-26.5%, unadjusted p = 0.94"),
    ("latitude", df["LATITUDE"], "numeric", "LATITUDE", "Latitude", "AUC 0.501"),
    ("longitude", df["LONGITUDE"], "numeric", "LONGITUDE", "Longitude", "AUC 0.500"),
    ("in_venue", in_venue, "binary", "RESTAURANT_LOCATION vs RESTAURANT_NAME",
     "1 = located inside a larger venue (location name differs from the restaurant name)", "24.0% vs 23.4%, unadjusted p = 0.26"),
]:
    add_feature(name, values, "location", kind, source=source, description=description,
                rationale="Neighborhood or venue type could plausibly relate to risk")
    drop_feature(name, f"EDA finding 14: no clear association ({evidence})")

categories kept: 12 + 'Other' (734 training rows pooled); missing: 177 training rows


**Operating hours.** The column holds 24 distinct schedule strings in a handful of formats (day ranges, split shifts, overnight hours, "24 hours", "Closed …"). `parse_schedule()` turns each one into the half-hour slots of the week it's open. The parsed table is printed in full so every schedule can be checked by eye.

In [8]:
DAY_INDEX = {day: i for i, day in enumerate(["mon", "tue", "wed", "thu", "fri", "sat", "sun"])}
DAY_LIST = re.compile(r"(mon|tue|wed|thu|fri|sat|sun)(-(mon|tue|wed|thu|fri|sat|sun))?")
SLOTS_PER_DAY = 48  # half-hour resolution ('10:30am')


def clock(text):
    """'10am' / '10:30pm' / '12am' -> hours after midnight (12am = 0, 12pm = 12)."""
    hour, minute, half = re.fullmatch(r"(\d{1,2})(?::(\d\d))?(am|pm)", text).groups()
    return int(hour) % 12 + (12 if half == "pm" else 0) + int(minute or 0) / 60


def parse_schedule(schedule):
    """Open half-hour slots of the week (slot 0 = Monday 00:00) and the set of days with an opening.
    A part without a day list repeats the previous one (every day at the start, e.g. 'Lunch ..., Dinner ...');
    a time range that ends at or before its start runs past midnight; 'Closed ...' parts add nothing."""
    slots, days, opening_days = np.zeros(7 * SLOTS_PER_DAY, dtype=bool), list(range(7)), set()
    for part in schedule.lower().split(", "):
        if part.startswith("closed"):
            continue
        head, _, rest = part.partition(" ")
        if DAY_LIST.fullmatch(head):
            first, _, last = head.partition("-")
            days, part = list(range(DAY_INDEX[first], DAY_INDEX[last or first] + 1)), rest
        elif head in ("daily", "lunch", "dinner"):
            days, part = (list(range(7)) if head == "daily" else days), rest
        if part == "24 hours":
            start, length = 0.0, 24.0
        else:
            start, end = map(clock, part.split("-"))
            length = (end - start) % 24 or 24.0
        for day in days:
            first_slot = day * SLOTS_PER_DAY + int(start * 2)
            slots[np.arange(first_slot, first_slot + int(length * 2)) % slots.size] = True
            opening_days.add(day)
    return slots, opening_days


def schedule_attributes(schedule):
    slots, opening_days = parse_schedule(schedule)
    by_day = slots.reshape(7, SLOTS_PER_DAY)
    return {"hours_per_week": slots.sum() / 2, "days_open": len(opening_days),
            "open_late": float(by_day[:, : 4 * 2].any()),  # open at some point between midnight and 4 am
            "open_24h": float(slots.all())}


hours_table = pd.DataFrame({s: schedule_attributes(s) for s in df["restaurant_operating_hours"].dropna().unique()}).T
HOURS = {
    "hours_per_week": ("numeric", "Opening hours per week"),
    "days_open": ("numeric", "Days per week with an opening"),
    "open_late": ("binary", "1 = open at some point between midnight and 4 am"),
    "open_24h": ("binary", "1 = open 24 hours, 7 days a week"),
}
for name, (kind, description) in HOURS.items():
    add_feature(name, df["restaurant_operating_hours"].map(hours_table[name]), "restaurant", kind,
                source="restaurant_operating_hours (parsed)", description=description,
                rationale="EDA finding 14: the 24 schedules differ only borderline (unadjusted p = 0.045) with no "
                          "pattern; parsed into interpretable attributes instead of 24 one-hot columns so section 9 "
                          "can test them")
hours_table.sort_values("hours_per_week")

,hours_per_week,days_open,open_late,open_24h
"Fri-Sat 10pm-4am, Thu 10pm-2am",16.0,3.0,1.0,0.0
"Thu-Sun 5pm-12am, Closed Mon-Wed",28.0,4.0,0.0,0.0
"Wed-Sun 4pm-11pm, Closed Mon-Tue",35.0,5.0,0.0,0.0
"Tue-Sun 5pm-12am, Closed Mon",42.0,6.0,0.0,0.0
"Mon-Fri 11am-2pm, 5pm-10pm, Sat-Sun 5pm-10pm",50.0,7.0,0.0,0.0
Daily 6pm-2am,56.0,7.0,1.0,0.0
"Lunch 11am-2pm, Dinner 5pm-10pm",56.0,7.0,0.0,0.0
"Tue-Sun 11am-9pm, Closed Mon",60.0,6.0,0.0,0.0
"Mon-Fri 10am-9pm, Sat-Sun 11am-10pm",77.0,7.0,0.0,0.0
"Mon-Thu 10am-9pm, Fri-Sat 10am-11pm, Sun 10am-9pm",81.0,7.0,0.0,0.0


## 6. The added business columns

EDA section 9 found no meaningful association for any of the seven added business columns (univariate AUC 0.49–0.51). The largest effect is complaints: the top quintile is about 2 points higher (unadjusted p = 0.014). They're registered here. Section 9 tests them, as one block and one at a time, before the final decision.

In [9]:
BUSINESS = {
    "annual_customer_complaint_count": ("numeric", "Customer complaints per year"),
    "foodborne_illness_count": ("numeric", "Foodborne illnesses on record"),
    "review_rating": ("numeric", "Online review rating"),
    "cleaning_station_count": ("numeric", "Cleaning stations on site"),
    "allergen_labeling_flag": ("binary", "1 = allergen labeling in place"),
    "inspection_announced_flag": ("binary", "1 = the inspection was announced in advance"),
    "safety_certification_flag": ("binary", "1 = safety certification on file"),
}
for column, (kind, description) in BUSINESS.items():
    add_feature(column, df[column], "business", kind, source=f"{column} (added business column)",
                description=description,
                rationale="Intuitively linked to food-safety risk, so tested despite the EDA's weak first look")
describe_features(list(BUSINESS))

,missing,mean,min,max
annual_customer_complaint_count,0,51.445,1.0,419.0
foodborne_illness_count,0,0.611,0.0,12.0
review_rating,0,4.088,2.0,5.0
cleaning_station_count,0,3.002,1.0,5.0
allergen_labeling_flag,0,0.708,0.0,1.0
inspection_announced_flag,0,0.277,0.0,1.0
safety_certification_flag,0,0.625,0.0,1.0


## 7. Comment features

How the provided lookups fit together: the **acronym lookup** turns shorthand into words, the **regulations corpus** says which food-safety topic a fragment is about, and topic-less fragments are the chatter to ignore. Every decision is made on **templates** (fragments with numbers masked as `#`), using training rows only.

### 7a. Expand acronyms

This makes two fixes over a plain whole-word match, both flagged in EDA section 11:
- Numbers are masked *before* expanding, so `150ppm` becomes `#ppm` and its `ppm` is found. Lookup keys containing digits (`3-COMP`, `2-DR`) are masked the same way.
- A plural `s` is allowed, so `hh cards` matches `HH CARD`.

In [10]:
def template_of(fragment):
    """Fragment with every number masked as '#': 'wic @ 45f' -> 'wic @ #f'."""
    return re.sub(r"\d+", "#", fragment)


def acronym_map(table):
    """lowercase masked acronym -> meaning. 'NCA / NCIAA' becomes two entries; '3-COMP' becomes '#-comp'."""
    mapping = {}
    for acronym, meaning in zip(table["Acronym"], table["Meaning"]):
        for key in acronym.split(" / "):
            mapping[template_of(key.strip().lower())] = meaning.lower()
    return mapping


ACRONYMS = acronym_map(acronyms)
ACRONYM_RE = re.compile(r"(?<![\w/-])(" + "|".join(sorted(map(re.escape, ACRONYMS), key=len, reverse=True))
                        + r")s?(?![\w/-])")


def expand_acronyms(text):
    """'no soap @ front hws' -> 'no soap @ front handwash sink / station'."""
    return ACRONYM_RE.sub(lambda m: ACRONYMS[m.group(1)], text)


fragments = get_fragments(df)
train_templates = fragments[is_train].explode().dropna().map(template_of)
EXPANDED = {t: expand_acronyms(t) for t in train_templates.unique()}
changed = pd.DataFrame({"template": list(EXPANDED), "expanded": list(EXPANDED.values())}).query("template != expanded")
print(f"{len(EXPANDED)} distinct training templates; {len(changed)} contain at least one lookup acronym")
changed.sample(12, random_state=RANDOM_STATE)

307 distinct training templates; 69 contain at least one lookup acronym


,template,expanded
80,mop sink ok,mop/utility sink ok
1,# viol noted,# violation noted
170,uncovered pans in ric,uncovered pans in reach-in cooler
14,reinsp req #d,re-inspection req #d
198,signed off,report signed by pic
60,pic uncoop,person in charge uncoop
39,pic ok,person in charge ok
113,grease trap due,grease interceptor/trap due
165,raw over rte wic,raw over ready-to-eat (food) walk-in cooler
44,dry stg orderly,dry storage orderly


### 7b. Map fragments to regulation topics

The 26 curated corpus documents each have a `topic` and quote the inspectors' shorthand. `topic_similarity()` scores every training template against every topic, using TF-IDF fitted **on the corpus only** (no inspection rows, no target, so no leakage).

In [11]:
curated = corpus[corpus["topic"] != "violation_code_definition"]
topic_text = (curated["title"] + ". " + curated["text"]).groupby(curated["topic"]).agg(" ".join)


def topic_similarity(texts, topic_text):
    """Best-matching regulation topic and its cosine similarity for each text (vectorizer fitted on the corpus only)."""
    vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), sublinear_tf=True).fit(topic_text)
    sims = cosine_similarity(vectorizer.transform(texts), vectorizer.transform(topic_text))
    return pd.DataFrame({"topic": topic_text.index[sims.argmax(axis=1)], "similarity": sims.max(axis=1)})


template_table = train_templates.value_counts().rename("train_fragments").rename_axis("template").reset_index()
template_table["expanded"] = template_table["template"].map(EXPANDED)
template_table = template_table.join(topic_similarity(template_table["expanded"], topic_text))
print(f"{len(template_table)} training templates scored against {len(topic_text)} corpus topics; strongest matches:")
template_table.sort_values("similarity", ascending=False).head(40)

307 training templates scored against 25 corpus topics; strongest matches:


,template,train_fragments,expanded,topic,similarity
79,nca compliant,1909,nevada clean indoor air act compliant,clean_indoor_air,0.569223
90,chlor test kit missing,1895,chlorine sanitizer test kit missing,test_kits,0.449844
62,cert fpm exp #/#,1960,cert food protection manager exp #/#,cfpm,0.414603
111,tphc log spotty,1869,time as a public health control log spotty,time_as_control,0.392578
15,tcs @ #f on line,4417,time/temperature control for safety (food) @ #f on line,cold_hot_holding,0.375586
113,tcs temped ok,1866,time/temperature control for safety (food) temped ok,cold_hot_holding,0.375586
54,no hh cards x#,2068,no food handler health card x#,food_handler_cards,0.373573
56,tcs left out ~#hr,2019,time/temperature control for safety (food) left out ~#hr,cold_hot_holding,0.342338
93,date mk missing,1892,date marking missing,date_marking,0.329692
127,gaskets torn wic,1851,door gaskets/seals torn walk-in cooler,equipment_condition,0.326454


**Reviewing the matches.** Above a similarity of about 0.20, the corpus match is right for most templates. Below it, the scores are noise: chatter containing the word "time" scores 0.22 against *time as a public health control*, while real observations written in pure shorthand ("cockroach egg casings", "meat slicer moldy") score 0 because they share no words with the corpus. So the threshold is set at **0.20**, and all 307 templates were reviewed by hand. EDA section 11 had already sorted the same templates into observations, process notes, restated business facts, chatter and redacted remarks. The overrides below fix three kinds of error:

1. **Above the threshold but not a food-safety observation** (chatter, restated facts, the posted grade): no topic.
2. **Above the threshold but the wrong topic:** corrected.
3. **Food-safety observations below the threshold:** given their topic by hand.

Three labels are added for observation notes that no corpus topic covers: `corrective_action` (critical or priority items corrected or flagged for follow-up), `person_in_charge` (notes on the manager on duty) and `general_condition` (overall praise such as "spotless prep area"). Templates that end up without a topic are treated as chatter.

In [12]:
SIMILARITY_THRESHOLD = 0.20  # at or above it, the corpus match is accepted unless overridden below

TOPIC_OVERRIDES = {
    # 1. matched above the threshold, but not a food-safety observation -> no topic
    "safety training documented": None,                       # restates safety_certification_flag
    "tot dem #": None,                                        # process note (total demerits)
    "grade a posted": None, "grade b posted": None, "grade c posted": None,  # restate CURRENT_GRADE (section 4)
    "owner nervous the whole time": None, "felt like i was in a time capsule": None,  # chatter matched on "time"
    "long-time": None, "spouse texted asking what time i'll be h": None,               # truncated chatter
    "owner went full [redacted] about the demerits": None, "owner [redacted] when i mentioned the demerits": None,
    # 2. matched above the threshold, wrong topic
    "dish mach no sani": "warewashing_sanitizing", "sani low-retested": "warewashing_sanitizing",
    "sani @ #ppm quat": "warewashing_sanitizing", "quat @ #ppm low": "warewashing_sanitizing",  # readings, not kits
    # 3. food-safety observations below the threshold
    "no soap @ front hws": "handwashing", "hws blocked by cart": "handwashing", "hw stn used for storage": "handwashing",
    "hwash stn blocked": "handwashing", "hws no ppr twls": "handwashing", "no hot water handsink": "handwashing",
    "diligent handwashing observed": "handwashing",
    "hot hold @ #f ok": "cold_hot_holding", "hot hold @ #f": "cold_hot_holding", "wic @ #f (>#)": "cold_hot_holding",
    "wic @ #f": "cold_hot_holding", "ric @ #f": "cold_hot_holding", "cold hold @ #f": "cold_hot_holding",
    "phf ok": "cold_hot_holding", "temperatures all in range": "cold_hot_holding",
    "stem thermo calib ok": "thermometers",
    "uncovered pans in ric": "cross_contamination", "chem nr food prep": "chemicals",
    "scoops no handle": "in_use_utensils", "to-go cups on floor": "in_use_utensils",
    "emp drink on prep tbl": "employee_health",
    "#-comp not set up": "warewashing_sanitizing", "#-comp sink ok": "warewashing_sanitizing",
    "vermin harborage": "pest_control", "cockroach egg casings": "pest_control", "roaches near cook line": "pest_control",
    "ant traps": "pest_control",
    "sewage backup floor drain": "plumbing_sewage", "raw sewage odor": "plumbing_sewage",
    "grease trap due": "plumbing_sewage",
    "expired food on line": "date_marking", "labels missing bulk bins": "approved_source",
    "meat slicer moldy": "equipment_condition", "mold in ice machine": "equipment_condition",
    "ice mach clean": "equipment_condition", "grease buildup hood": "equipment_condition",
    "#-dr reach-in ok": "equipment_condition",
    "flr drk btwn cook line": "facility_maintenance", "ceiling stain wic": "facility_maintenance",
    "grout coming up": "facility_maintenance", "dry stg orderly": "facility_maintenance",
    "mop bleacher": "facility_maintenance", "mop sink ok": "facility_maintenance", "back dock ok": "facility_maintenance",
    # observation notes no corpus topic covers (labels added here)
    "**crit-corrected**": "corrective_action", "crit item, cos": "corrective_action", "cos": "corrective_action",
    "corrected during insp": "corrective_action", "priority viol-f/u": "corrective_action",
    "pic ok": "person_in_charge", "pic coop": "person_in_charge", "pic uncoop": "person_in_charge",
    "pic new mgr": "person_in_charge",
    "well maintained facility": "general_condition", "meticulous": "general_condition",
    "impeccably clean kitchen": "general_condition", "spotless prep area": "general_condition",
    "pristine dish area": "general_condition", "exemplary food handling": "general_condition",
}
unknown = set(TOPIC_OVERRIDES) - set(template_table["template"])
assert not unknown, f"overrides for templates that aren't in the training comments: {unknown}"

above = template_table["similarity"] >= SIMILARITY_THRESHOLD
overridden = template_table["template"].isin(list(TOPIC_OVERRIDES))
template_table["final_topic"] = np.where(overridden, template_table["template"].map(TOPIC_OVERRIDES),
                                         template_table["topic"].where(above))
TOPIC_OF = {t: (topic if isinstance(topic, str) else None)
            for t, topic in zip(template_table["template"], template_table["final_topic"])}
has_topic = template_table["template"].map(TOPIC_OF).notna()
share = lambda mask: template_table.loc[mask, "train_fragments"].sum() / template_table["train_fragments"].sum()

print(f"at or above {SIMILARITY_THRESHOLD}: {above.sum()} templates; match kept for {(above & ~overridden).sum()}, "
      f"overridden for {(above & overridden).sum()}")
print(f"given a topic by hand below the threshold: {(~above & has_topic).sum()} templates")
print(f"final: {has_topic.sum()} food-safety templates ({share(has_topic):.0%} of training fragments); "
      f"{(~has_topic).sum()} templates treated as chatter ({share(~has_topic):.0%} of training fragments)")
print("largest chatter templates:", "; ".join(template_table[~has_topic].nlargest(8, "train_fragments")["template"]))

(template_table[has_topic].groupby("final_topic")
 .agg(templates=("template", "size"), train_fragments=("train_fragments", "sum"),
      examples=("template", lambda s: "; ".join(s[:4])))
 .sort_values("train_fragments", ascending=False))

at or above 0.2: 40 templates; match kept for 25, overridden for 15
given a topic by hand below the threshold: 59 templates
final: 88 food-safety templates (41% of training fragments); 219 templates treated as chatter (59% of training fragments)
largest chatter templates: viol## noted; # viol noted; rtn visit; routine; re-insp f/u; routine insp; grade a posted; ch#


,templates,train_fragments,examples
final_topic,,,
cold_hot_holding,11,29384,hot hold @ #f ok; wic @ #f (>#); tcs @ #f on line; wic @ #f
handwashing,7,23697,no soap @ front hws; hws blocked by cart; hw stn used for storage; diligent handwashin...
corrective_action,5,18238,"priority viol-f/u; **crit-corrected**; crit item, cos; corrected during insp"
warewashing_sanitizing,7,17975,sani low-retested; #-comp not set up; dish mach no sani; quat @ #ppm low
pest_control,9,16948,flies at bar drain; vermin harborage; cockroach egg casings; ant traps
equipment_condition,7,14874,meat slicer moldy; mold in ice machine; ice mach clean; hood filters gunky
general_condition,6,14800,well maintained facility; meticulous; impeccably clean kitchen; spotless prep area
facility_maintenance,8,14669,mop bleacher; back dock ok; flr drk btwn cook line; dry stg orderly
cross_contamination,5,12922,raw chicken over produce; raw over rte wic; uncovered pans in ric; raw chx over rte


### 7c. Comment features

These features use only the food-safety fragments mapped above, so chatter, restated facts, process notes and redacted remarks can't trigger them (EDA findings 10–11). Each flag is defined by what the note *says*, and no algorithm fits it, but the choice of flags is **target-informed**: it follows the EDA's training-row rates (finding 9), and while building them the pest/sewage split and the critical-item definition were checked against training rates. Their value in section 9 is therefore part of the development results, not an independent test.
- **`pest_note`**: evidence of pests (roaches, flies, gnats, ants, rodents, droppings, vermin). "Glue traps empty" records a monitoring device, not pests, so it doesn't count.
- **`sewage_note`**: a sewage backup or sewage odor. It's kept separate from pests because the two add up (table below).
- **`critical_item_note`**: the note names a critical or priority item ("\*\*crit-corrected\*\*", "crit item, cos", "priority viol-f/u").
- **`positive_note`**: praise of the condition or practices ("well maintained facility", "spotless prep area", "temperatures all in range", …), as opposed to routine "ok" checks.
- **`comment_blank`**: no comment at all (8.1% of rows, EDA finding 12). It also tells the model that the flags above are 0 because there was nothing to read, not because a comment lacked such a note.
- **`comment_text`** (`kind="text"`): the food-safety templates, acronym-expanded and joined into one string. 04 turns it into TF-IDF inside CV for logistic regression. Temperature and ppm numbers stay masked, because EDA finding 11 showed they contradict their own "ok"/"low" labels.

Restated business facts contradict the structured columns, and the "grade X posted" notes restate `CURRENT_GRADE` (section 4), so neither is used.

In [13]:
def has_any(fragment_lists, patterns):
    """1 if any fragment in the row matches any of the regex patterns, else 0."""
    rx = re.compile("|".join(patterns))
    return fragment_lists.map(lambda frags: int(any(rx.search(f) for f in frags)))


# Each row's food-safety templates (unique, in order). Templates never seen in training have no topic and drop out.
food_safety = fragments.map(lambda frags: [t for t in dict.fromkeys(map(template_of, frags)) if TOPIC_OF.get(t)])
FOOD_SAFETY_SOURCE = "Comments -> food-safety templates (7b)"

COMMENT_FLAGS = {
    "pest_note": ([r"roach", r"flies", r"gnats", r"ant traps", r"rodent", r"droppings", r"vermin"],
                  "Evidence of pests: roaches, flies, gnats, ants, rodents, droppings or vermin",
                  "EDA findings 9-10: pest notes add risk within every violation-count band; the corpus maps them "
                  "to pest_control"),
    "sewage_note": ([r"sewage"], "Sewage backup or sewage odor",
                    "EDA finding 9: among the highest-rate observations (29.5-31.0%); the corpus treats a sewage "
                    "backup as an imminent health hazard"),
    "critical_item_note": ([r"crit", r"priority"],
                           "The inspector noted a critical or priority item (corrected on site or for follow-up)",
                           "EDA finding 9: '**crit-corrected**', 'crit item, cos' and 'priority viol-f/u' sit at "
                           "30-32%, against a 23.7% average"),
    "positive_note": ([r"well maintained", r"meticulous", r"impeccabl", r"spotless", r"pristine", r"exemplary",
                       r"all in range", r"thorough", r"diligent", r"orderly"],
                      "The inspector praised the condition or practices (e.g. 'spotless prep area')",
                      "EDA finding 9: praise notes sit at 17-19%, well below the 23.7% average"),
}
for name, (patterns, description, rationale) in COMMENT_FLAGS.items():
    add_feature(name, has_any(food_safety, patterns), "comments", "binary", source=FOOD_SAFETY_SOURCE,
                description=description, rationale=rationale)
add_feature("comment_blank", fragments.map(len).eq(0), "comments", "binary", source="Comments",
            description="1 = no inspector comment",
            rationale="EDA finding 12: 8.1% of rows, slightly higher rate (25.4% vs 23.5%); also marks the comment "
                      "flags as 'nothing to read' rather than 'no such note'")
add_feature("comment_text", food_safety.map(lambda ts: " . ".join(EXPANDED[t] for t in ts)), "comments", "text",
            source=FOOD_SAFETY_SOURCE + ", acronym-expanded",
            description="The inspection's food-safety notes as one cleaned string (empty when there are none)",
            rationale="EDA finding 10: only observation fragments carry signal (40% of templates beyond chance vs "
                      "0-6% for the rest); lets 04's TF-IDF use wording the flags don't cover")

# Check (training rows): how often each flag fires and the rate with and without it
y_tr = df.loc[is_train, TARGET]
flags = features.loc[is_train, list(COMMENT_FLAGS) + ["comment_blank"]]
print(pd.DataFrame({"rows flagged": flags.sum().astype(int), "share": flags.mean(),
                    "rate if 1": flags.apply(lambda f: y_tr[f.eq(1)].mean()),
                    "rate if 0": flags.apply(lambda f: y_tr[f.eq(0)].mean())}).round(3).to_string())
print("\npest and sewage notes add up (training rows):")
print(y_tr.groupby([flags["pest_note"], flags["sewage_note"]]).agg(rows="size", rate="mean").round(3).to_string())
print("\nexample comment_text:", features.loc[is_train, "comment_text"].iloc[0])

                    rows flagged  share  rate if 1  rate if 0
pest_note                  12505  0.446      0.267      0.213
sewage_note                 6112  0.218      0.296      0.220
critical_item_note         13121  0.468      0.310      0.172
positive_note              16109  0.575      0.209      0.274
comment_blank               2272  0.081      0.254      0.235

pest and sewage notes add up (training rows):
                        rows   rate
pest_note sewage_note              
0.0       0.0          12411  0.200
          1.0           3120  0.263
1.0       0.0           9513  0.247
          1.0           2992  0.331

example comment_text: no soap @ front handwash sink / station . **crit-corrected** . date marking missing . well maintained facility


## 8. Restaurant history

EDA section 10 found that repeat inspections of a restaurant are a median of 476 days apart and only mildly related. The data hold a *sample* of each restaurant's inspections, not its full history, so "earlier inspections" means earlier inspections **in this dataset**. Following the EDA's advice, history is built from earlier **inspection results** (violations), not earlier labels. The previous row's label describes a "next inspection" whose date isn't in the data, so it can't be confirmed to come before this one.

Each restaurant's whole history sits in one split (the holdout is grouped by restaurant), so history never crosses between training and holdout. The check below confirms the features only look backwards: recomputing them from the earlier half of the rows alone gives the same values.

In [14]:
def prior_features(frame, columns=()):
    """Per inspection: number of earlier inspections of the same restaurant, days since the previous one, and the mean
    of each column over earlier inspections only. Rows without an inspection date get NaN."""
    dated = frame[frame[TIME_COL].notna()].sort_values([GROUP_KEY, TIME_COL])
    by_restaurant = dated.groupby(GROUP_KEY)
    out = pd.DataFrame(index=dated.index)
    out["prior_inspections"] = by_restaurant.cumcount()
    out["days_since_prior"] = (dated[TIME_COL] - by_restaurant[TIME_COL].shift(1)).dt.days
    for col in columns:
        x = pd.to_numeric(dated[col], errors="coerce")
        filled, present = x.fillna(0), x.notna().astype(int)
        prior_sum = filled.groupby(dated[GROUP_KEY]).cumsum() - filled
        prior_n = present.groupby(dated[GROUP_KEY]).cumsum() - present
        out[f"prior_mean_{col}"] = prior_sum / prior_n.replace(0, np.nan)
    return out.reindex(frame.index)


HISTORY_INPUTS = ["n_violations", "critical_count"]
with_counts = df.assign(critical_count=features["critical_count"])
history = prior_features(with_counts, HISTORY_INPUTS)

# Backward-only check: the rows before the median date, on their own, must get exactly the same history values
cutoff = df[TIME_COL].median()
early = df[TIME_COL] < cutoff
history_backward_only = np.allclose(prior_features(with_counts[early], HISTORY_INPUTS).to_numpy(float),
                                    history[early].to_numpy(float), equal_nan=True)
print(f"history recomputed from the {early.sum():,} rows before {cutoff:%d %b %Y} alone is identical: "
      f"{history_backward_only}")

HISTORY = {
    "prior_inspections": ("prior_inspections", "Earlier inspections of this restaurant in the data"),
    "days_since_prior": ("days_since_prior", "Days since this restaurant's previous inspection in the data"),
    "prior_mean_violations": ("prior_mean_n_violations", "Mean violations over this restaurant's earlier inspections"),
    "prior_mean_critical": ("prior_mean_critical_count",
                            "Mean critical codes over this restaurant's earlier inspections"),
}
for name, (column, description) in HISTORY.items():
    add_feature(name, history[column], "history", "numeric",
                source=f"earlier rows of the same restaurant ({column})", description=description,
                rationale="EDA finding 15: mild repeat dependence; backward-looking summaries of earlier results "
                          "use it without relying on earlier labels")
describe_features(list(HISTORY))

history recomputed from the 17,341 rows before 28 Jun 2023 alone is identical: True


,missing,mean,min,max
prior_inspections,254,0.683,0.0,5.0
days_since_prior,16096,604.078,0.0,2343.0
prior_mean_violations,16097,5.331,3.0,22.0
prior_mean_critical,16155,1.176,0.0,7.0


## 9. Screen, check redundancy, and select (training rows only)

1. `screen()`: how well each feature alone ranks the target.
2. `high_correlations()`: near-duplicate pairs.
3. **Added value.** The selection starts from a *core* of features the EDA supports directly. `forward_select()` then adds the remaining candidates one at a time, best first, for as long as the best one raises the cross-validated ROC-AUC of a plain logistic regression by **at least 0.001 on average and in at least 4 of 5 folds**. Folds are grouped by restaurant, and imputation, scaling and one-hot encoding are fitted inside each fold, as in 04.
4. **Drop-one check.** A core feature stays only if removing it from the final set lowers the CV ROC-AUC on average **and** in at least 4 of 5 folds. Core features need a consistent contribution but no minimum size, because the EDA already showed their effect; candidates need both.
5. `drop_feature()` for everything left out.

The selection sees training rows only, and the holdout stays untouched. It is still **target-informed**: it uses the training labels on the same restaurant-grouped folds that 04 reuses for its comparisons and tuning, so CV scores of the selected set are somewhat optimistic. Only the held-out restaurants give an independent final evaluation. Logistic regression is linear, so a candidate whose signal is purely non-linear could be under-rated. The EDA's rate tables, which aren't linear, back each decision.

In [15]:
def screen(columns):
    """Single-feature ranking power on training rows (0.5 = none)."""
    rows = []
    y = df.loc[is_train, TARGET]
    for col in columns:
        x = pd.to_numeric(features.loc[is_train, col], errors="coerce")
        ok = x.notna()
        if ok.sum() >= 100 and x[ok].nunique() > 1:
            auc = roc_auc_score(y[ok], x[ok])
            rows.append({"feature": col, "auc": auc, "strength": max(auc, 1 - auc), "missing": 1 - ok.mean()})
    return pd.DataFrame(rows).set_index("feature").sort_values("strength", ascending=False)


def high_correlations(columns, threshold=0.85):
    """Pairs of features whose |Spearman correlation| on training rows is at least `threshold`."""
    corr = features.loc[is_train, columns].apply(pd.to_numeric, errors="coerce").corr(method="spearman").abs()
    upper = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1))
    pairs = upper.stack().dropna()
    return pairs[pairs >= threshold].sort_values(ascending=False).rename("abs_spearman").reset_index()


numeric_features = [n for n, r in registry.items() if r["kind"] in ("numeric", "binary")]
screened = screen(numeric_features)
screened.round(3)

,auc,strength,missing
feature,,,
first_violation_demerits,0.641,0.641,0.000
code_demerits_total,0.635,0.635,0.009
critical_count,0.626,0.626,0.009
n_violations,0.626,0.626,0.000
inspection_demerits,0.604,0.604,0.009
critical_item_note,0.595,0.595,0.000
seating_capacity,0.590,0.590,0.000
total_mapped_demerits,0.590,0.590,0.009
large_venue,0.589,0.589,0.000


In [16]:
high_correlations([n for n in numeric_features if registry[n]["keep"]])

,level_0,level_1,abs_spearman
0,critical_count,critical_share,0.904231


In [17]:
y_train = df.loc[is_train, TARGET]
SCREEN_FOLDS = list(GroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
                    .split(features.loc[is_train], y_train, groups=df.loc[is_train, GROUP_KEY]))
MIN_GAIN, MIN_FOLDS = 0.001, 4


def fold_aucs(columns):
    """ROC-AUC on each grouped fold for a plain logistic regression on `columns` (training rows only; median
    imputation with missing-value indicators, scaling and one-hot encoding are fitted inside each fold)."""
    numeric = [c for c in columns if registry[c]["kind"] in ("numeric", "binary")]
    categorical = [c for c in columns if registry[c]["kind"] == "categorical"]
    prep = ColumnTransformer([
        ("num", make_pipeline(SimpleImputer(strategy="median", add_indicator=True), StandardScaler()), numeric),
        ("cat", make_pipeline(SimpleImputer(strategy="constant", fill_value="missing"),
                              OneHotEncoder(handle_unknown="ignore")), categorical)])
    model = make_pipeline(prep, LogisticRegression(max_iter=2000))
    X = features.loc[is_train, columns]
    scores = []
    for fit_idx, valid_idx in SCREEN_FOLDS:
        fitted = clone(model).fit(X.iloc[fit_idx], y_train.iloc[fit_idx])
        scores.append(roc_auc_score(y_train.iloc[valid_idx], fitted.predict_proba(X.iloc[valid_idx])[:, 1]))
    return np.array(scores)


def compare(base_scores, variants):
    """CV ROC-AUC of each variant (name -> column list), its mean change from the base, and how many folds improved."""
    rows = {}
    for name, columns in variants.items():
        s = fold_aucs(columns)
        rows[name] = {"cv_auc": s.mean(), "change": (s - base_scores).mean(), "folds_better": int((s > base_scores).sum())}
    return pd.DataFrame(rows).T.astype({"folds_better": int})


def forward_select(core, candidates):
    """Greedy forward selection from `core`: add the candidate that raises the CV ROC-AUC most, while that gain is at
    least MIN_GAIN on average and positive in at least MIN_FOLDS folds.
    Returns (added candidates, log of each step, last round: what every left-out candidate would still add)."""
    selected, remaining, steps = list(core), list(candidates), []
    current = fold_aucs(selected)
    while remaining:
        round_table = compare(current, {c: selected + [c] for c in remaining}).sort_values("change", ascending=False)
        best = round_table.index[0]
        passes = round_table.loc[best, "change"] >= MIN_GAIN and round_table.loc[best, "folds_better"] >= MIN_FOLDS
        steps.append({"best candidate": best, **round_table.loc[best].to_dict(), "added": passes})
        if not passes:
            break
        selected.append(best)
        remaining.remove(best)
        current = fold_aucs(selected)
    log = pd.DataFrame(steps, index=pd.RangeIndex(1, len(steps) + 1, name="step")).astype({"folds_better": int})
    return [c for c in selected if c not in core], log, round_table

**Core and candidates.** The core holds the features each EDA table supports on its own:
- the violation count, the critical count, worst severity and the IHH flag (findings 1–4), with `violation_list_complete`;
- `large_venue` and `is_buffet` (findings 6 and 8);
- the four comment flags and `comment_blank` (findings 9–12).

Everything else registered and not yet dropped is a candidate:
- the overlapping demerit and severity measures;
- the pooled category;
- the parsed operating hours;
- the business columns;
- history.

`current_status` is deliberately left out of both. 04 decides it, so it's only measured here, for information.

In [18]:
CORE = ["n_violations", "violation_list_complete", "critical_count", "worst_severity_rank", "ihh_present",
        "large_venue", "is_buffet", "pest_note", "sewage_note", "critical_item_note", "positive_note", "comment_blank"]
CANDIDATES = (["code_demerits_total", "inspection_demerits", "first_violation_demerits", "major_count",
               "non_major_count", "critical_share", "restaurant_category"]
              + list(HOURS) + list(BUSINESS) + list(HISTORY))
assert all(registry[c]["keep"] for c in CORE + CANDIDATES)

core_scores = fold_aucs(CORE)
print(f"core ({len(CORE)} features): CV ROC-AUC {core_scores.mean():.4f} "
      f"(folds {core_scores.min():.4f} to {core_scores.max():.4f})")
added, selection_log, left_out = forward_select(CORE, CANDIDATES)
FINAL = CORE + added
final_scores = fold_aucs(FINAL)
print(f"added to the core: {', '.join(added) if added else 'nothing'} | final set ({len(FINAL)} features): "
      f"CV ROC-AUC {final_scores.mean():.4f}")
selection_log.round(4)

core (12 features): CV ROC-AUC 0.7097 (folds 0.7007 to 0.7169)


added to the core: first_violation_demerits | final set (13 features): CV ROC-AUC 0.7211


,best candidate,cv_auc,change,folds_better,added
step,,,,,
1,first_violation_demerits,0.7211,0.0114,5,True
2,inspection_demerits,0.7213,0.0002,4,False


In [19]:
# What each left-out candidate would still add on top of the final set, one at a time
left_out.round(4)

,cv_auc,change,folds_better
inspection_demerits,0.7213,0.0002,4
foodborne_illness_count,0.7212,0.0000,4
critical_share,0.7211,0.0000,3
allergen_labeling_flag,0.7211,-0.0000,2
code_demerits_total,0.7211,-0.0000,3
open_late,0.7211,-0.0000,3
annual_customer_complaint_count,0.7211,-0.0000,1
cleaning_station_count,0.7211,-0.0001,2
open_24h,0.7210,-0.0001,1
inspection_announced_flag,0.7210,-0.0001,0


In [20]:
# The same question for whole blocks, plus current_status for information (04 decides it)
blocks = {"business columns (all 7)": list(BUSINESS), "operating-hours attributes (all 4)": list(HOURS),
          "history (all 4)": list(HISTORY), "current status (both; decided in 04)": ["current_grade_rank", "current_demerits"]}
block_table = compare(final_scores, {name: FINAL + columns for name, columns in blocks.items()})
block_table.round(4)

,cv_auc,change,folds_better
business columns (all 7),0.7209,-0.0002,1
operating-hours attributes (all 4),0.7208,-0.0003,1
history (all 4),0.7207,-0.0004,0
current status (both; decided in 04),0.7206,-0.0005,0


In [21]:
# Drop-one check: how much the final set loses when each feature is removed
drop_one = compare(final_scores, {c: [f for f in FINAL if f != c] for c in FINAL})
drop_one = drop_one.rename(columns={"change": "change_when_removed", "folds_better": "folds_better_without"})
drop_one.sort_values("change_when_removed").round(4)

,cv_auc,change_when_removed,folds_better_without
large_venue,0.6956,-0.0256,0
n_violations,0.7094,-0.0117,0
first_violation_demerits,0.7097,-0.0114,0
positive_note,0.7160,-0.0051,0
sewage_note,0.7167,-0.0044,0
pest_note,0.7171,-0.0040,0
ihh_present,0.7196,-0.0015,0
critical_item_note,0.7200,-0.0011,0
comment_blank,0.7206,-0.0005,2
is_buffet,0.7207,-0.0004,1


**What the selection shows** (training rows, grouped 5-fold CV, plain logistic regression; these are development results used to choose features, not a final performance estimate):
- **The EDA core reaches a CV ROC-AUC of 0.710, and one candidate clears the bar.** `first_violation_demerits` adds +0.011, better in all 5 folds, for 0.721. No other candidate adds even +0.001; the next best is `inspection_demerits` at +0.0002.
- **`first_violation_demerits` is kept despite the EDA's caution.** It's recorded at this inspection, so it's available at prediction time, and its gain is too large and consistent to ignore. Why the *first* listed code matters beyond the counts and the worst severity isn't documented. The likeliest explanations are that inspectors list the most important finding first, or that the exercise data were built that way. 04's driver analysis (SHAP) should confirm it behaves sensibly (more demerits → more risk).
- **Once it's in, the overlapping measures add nothing:** `code_demerits_total`, `inspection_demerits`, the Major and Non-Major counts, and `critical_share` (which nearly duplicates `critical_count`, Spearman 0.90).
- **Business columns, operating hours and history add nothing, alone or as blocks.** This confirms EDA findings 13–15. The pooled category does worse than the buffet flag alone.
- **Drop-one check:**
  - `large_venue` costs the most to remove (−0.026), then `n_violations` and `first_violation_demerits` (about −0.011 each), then the comment flags (−0.001 to −0.005).
  - `critical_count` and `worst_severity_rank` add only about 0.0003 each because they overlap, but removing either lowers the AUC in every fold, so both stay.
  - `violation_list_complete` and `comment_blank` don't lower it consistently and are dropped below. `violation_list_complete` adds nothing because 04's imputer already adds a missing-value indicator to every list-based feature.
- **For 04:** `current_status` adds nothing on top of the final set (−0.0005, better in 0 of 5 folds). Together with the timing risk, that argues for leaving it out of the final model, which 04's with/without comparison can confirm.

`comment_text` isn't tested here because its TF-IDF has to be fitted inside 04's CV; 04 compares the model with and without it.

### Decisions

In [22]:
# 1. Every candidate the forward selection didn't add is dropped; the reason quotes what it would still add
EXTRA = {name: " EDA finding 13: univariate AUC 0.49-0.51." for name in BUSINESS}
EXTRA.update({name: " EDA finding 14: borderline across the 24 schedules (unadjusted p = 0.045), no pattern."
              for name in HOURS})
EXTRA.update({name: " EDA finding 15: weak repeat dependence; the data hold a sample of each restaurant's inspections."
              for name in HISTORY})
for name in CANDIDATES:
    if name not in added:
        row = left_out.loc[name]
        drop_feature(name, f"Adds {row['change']:+.4f} CV ROC-AUC on top of the final set (better in "
                           f"{int(row['folds_better'])} of 5 folds), below the bar of +{MIN_GAIN} in {MIN_FOLDS}+ folds."
                           + EXTRA.get(name, ""))

# 2. A core feature stays only if removing it lowers the CV ROC-AUC on average and in at least MIN_FOLDS folds
CORE_NOTE = {
    "violation_list_complete": " Its information is already in the missing-value indicators 04's imputer adds to "
                               "the list-based features (and in gradient boosting's own handling of NaN).",
    "comment_blank": " EDA finding 12 was a small effect (25.4% vs 23.5%).",
}
for name, row in drop_one.iterrows():
    if not (row["change_when_removed"] < 0 and row["folds_better_without"] <= 5 - MIN_FOLDS):
        drop_feature(name, f"Removing it from the final set changes CV ROC-AUC by {row['change_when_removed']:+.4f} "
                           f"and improves {int(row['folds_better_without'])} of 5 folds: no consistent contribution."
                           + CORE_NOTE.get(name, ""))

SELECTED = [f for f in FINAL if registry[f]["keep"]]
selected_scores = fold_aucs(SELECTED)
print(f"final selection ({len(SELECTED)} features; plus comment_text and current_status, which 04 tests): "
      f"CV ROC-AUC {selected_scores.mean():.4f} (folds {selected_scores.min():.4f} to {selected_scores.max():.4f})")

dictionary = pd.DataFrame(registry.values())
kept = dictionary[dictionary["keep"]]
print(f"{len(dictionary)} features registered: {len(kept)} kept, {len(dictionary) - len(kept)} dropped")
kept[["feature", "feature_set", "kind", "description", "rationale"]]

final selection (11 features; plus comment_text and current_status, which 04 tests): CV ROC-AUC 0.7206 (folds 0.7120 to 0.7240)
49 features registered: 14 kept, 35 dropped


,feature,feature_set,kind,description,rationale
0,n_violations,violations,numeric,Number of violations recorded at this inspection,EDA finding 1: the rate climbs steadily from 15.3% at 3 violations to 71.3% at 15+ wit...
2,critical_count,violations,numeric,Number of Critical codes recorded at this inspection,EDA finding 3: the strongest severity count; the rate rises from 14.4% with no critica...
3,worst_severity_rank,violations,numeric,"Most severe code recorded: 0 = no-demerit codes only, 1 = Non-Major, 2 = Major, 3 = Cr...","EDA finding 2: the rate steps up in the same order as the ranks (Non-Major 10.5%, Majo..."
4,ihh_present,violations,binary,1 = an Imminent Health Hazard code is confirmed; 0 = a complete list has none; missing...,EDA finding 4: a confirmed hazard raises the rate to 29.8% vs 23.3%. IHH codes carry 0...
9,first_violation_demerits,violations,numeric,Demerits of the first code on the inspection report,"EDA finding 5: the strongest single demerit measure (AUC 0.641), but the first code is..."
13,current_grade_rank,current_status,numeric,"Current letter grade as an ordered number: 0 = A, 1 = B, 2 = C","EDA finding 7: A 21.9%, B 25.8%, C 29.9%, an ordered step. LEAKAGE RISK: its timing is..."
14,current_demerits,current_status,numeric,The restaurant's current demerits,EDA finding 7: 18.5% at 0-2 demerits to 41.6% at 11+. LEAKAGE RISK: same timing caveat...
15,large_venue,restaurant,binary,1 = more than 200 seats,"EDA finding 6: 30.6% above 200 seats vs 17.6% at or below, flat within each side, and ..."
18,is_buffet,restaurant,binary,1 = Buffet,"EDA finding 8: the one category that clearly differs (14.3% [11.2, 18.2], n = 453); th..."
39,pest_note,comments,binary,"Evidence of pests: roaches, flies, gnats, ants, rodents, droppings or vermin",EDA findings 9-10: pest notes add risk within every violation-count band; the corpus m...


In [23]:
dictionary.loc[~dictionary["keep"], ["feature", "feature_set", "drop_reason"]]

,feature,feature_set,drop_reason
1,violation_list_complete,violations,Removing it from the final set changes CV ROC-AUC by -0.0000 and improves 2 of 5 folds...
5,code_demerits_total,violations,"Adds -0.0000 CV ROC-AUC on top of the final set (better in 3 of 5 folds), below the ba..."
6,major_count,violations,"Adds -0.0001 CV ROC-AUC on top of the final set (better in 1 of 5 folds), below the ba..."
7,non_major_count,violations,"Adds -0.0002 CV ROC-AUC on top of the final set (better in 1 of 5 folds), below the ba..."
8,critical_share,violations,"Adds +0.0000 CV ROC-AUC on top of the final set (better in 3 of 5 folds), below the ba..."
10,total_mapped_demerits,violations,EDA finding 5: covers only the first three codes (AUC 0.590) and barely tracks the rec...
11,inspection_demerits,violations,"Adds +0.0002 CV ROC-AUC on top of the final set (better in 4 of 5 folds), below the ba..."
12,is_reinspection,inspection,"EDA section 7: re-inspections 24.0% (n = 1,713) vs routine 23.6%, unadjusted p = 0.74"
16,seating_capacity,restaurant,EDA finding 6: no clear difference within <=200 seats (unadjusted p = 0.58) or within ...
17,restaurant_category,restaurant,"Adds -0.0006 CV ROC-AUC on top of the final set (better in 0 of 5 folds), below the ba..."


### Final selection

| Feature set | Kept features | Why |
|---|---|---|
| `violations` | `n_violations`, `critical_count`, `worst_severity_rank`, `ihh_present`, `first_violation_demerits` | What this inspection found: how many violations, how severe the worst one was, whether a hazard was confirmed, and the first code listed |
| `restaurant` | `large_venue`, `is_buffet` | Venues with more than 200 seats are much riskier at every violation count; buffets are lower |
| `comments` | `pest_note`, `sewage_note`, `critical_item_note`, `positive_note`, `comment_text` | Inspector notes carry information the violation codes don't |
| `current_status` | `current_grade_rank`, `current_demerits` | Kept only so 04 can run the with/without comparison; they add nothing on top of the features above, and their timing is uncertain |

Every other feature considered is in the dropped table above with its reason. Imputation, scaling, one-hot encoding and TF-IDF are all fitted later, inside 04's cross-validation pipeline.

## 10. Readiness checklist and save

Outputs are written only when every check passes (or set `SAVE_ANYWAY = True` to save work in progress).

In [24]:
FORBIDDEN = {TARGET, GROUP_KEY, "RESTAURANT_PERMIT_NUMBER", "RECORD_UPDATED"}


def readiness_checks():
    """Pass/fail for each goal; never raises."""
    kept = [r for r in registry.values() if r["keep"]]
    train_rows = features.loc[is_train]
    checks = {
        "features cover every row": len(features) == len(df),
        "more than the worked example registered": len(registry) > 1,
        "every feature column is documented": set(features.columns) == set(registry),
        "no target / ID / post-inspection column used as a feature": not (set(features.columns) & FORBIDDEN),
        "every feature has a rationale (no TODO left)":
            all(r["rationale"] and not r["rationale"].startswith("TODO") for r in registry.values()),
        "every dropped feature has a reason": all(r["keep"] or r["drop_reason"] for r in registry.values()),
        "numeric/binary features are numeric": all(pd.api.types.is_numeric_dtype(features[r["feature"]])
                                                   for r in kept if r["kind"] in ("numeric", "binary")),
        "no kept feature is constant or empty on training rows":
            all(train_rows[r["feature"]].nunique(dropna=True) > 1 for r in kept),
        "text features are strings (no NaN)": all(features[r["feature"]].map(lambda v: isinstance(v, str)).all()
                                                  for r in kept if r["kind"] == "text"),
        "exactly one text feature kept (04's TF-IDF takes one column)": sum(r["kind"] == "text" for r in kept) == 1,
        "current_status kept for 04's with/without comparison":
            any(r["feature_set"] == "current_status" for r in kept),
        "history features only look backwards (truncation check)": history_backward_only,
    }
    for name, ok in checks.items():
        print(("✅ " if ok else "❌ ") + name)
    return all(checks.values())


all_passed = readiness_checks()

✅ features cover every row
✅ more than the worked example registered
✅ every feature column is documented
✅ no target / ID / post-inspection column used as a feature
✅ every feature has a rationale (no TODO left)
✅ every dropped feature has a reason
✅ numeric/binary features are numeric
✅ no kept feature is constant or empty on training rows
✅ text features are strings (no NaN)
✅ exactly one text feature kept (04's TF-IDF takes one column)
✅ current_status kept for 04's with/without comparison
✅ history features only look backwards (truncation check)


In [25]:
SAVE_ANYWAY = False

if all_passed or SAVE_ANYWAY:
    FEAT.mkdir(exist_ok=True)
    table = df[[GROUP_KEY, TIME_COL, TARGET]].assign(split=split).join(features)
    table.to_pickle(FEAT / "features.pkl")
    pd.DataFrame(registry.values()).to_csv(FEAT / "feature_dictionary.csv", index=False)
    print(f"saved {features.shape[1]} features ({len(kept)} kept) to {FEAT}")
    print(kept.groupby("feature_set")["feature"].agg(list).to_string())
else:
    print("Not saved: finish the failing checks above, or set SAVE_ANYWAY = True.")

saved 49 features (14 kept) to C:\Users\19176\las-vegas-restaurant-inspections\features
feature_set
comments                          [pest_note, sewage_note, critical_item_note, positive_note, comment_text]
current_status                                                       [current_grade_rank, current_demerits]
restaurant                                                                         [large_venue, is_buffet]
violations        [n_violations, critical_count, worst_severity_rank, ihh_present, first_violation_demer...
